# BiciTierra Market - Segmentacio

Objectiu: preparar una lectura útil de perfils de client abans de passar a un clustering més formal.

## Dades

Hi ha 2.000 clients sintètics, una fotografia de 30/06/2026 independent de la taula de vendes. Per a un clustering, excloeu identificador i data, escaleu magnituds i decidiu com tractar els nuls. `annual_spend`, `avg_order_value` i `orders_last_12m` tenen una relació aritmètica: usar-les totes pot donar massa pes al valor monetari. La segmentació per regles és només una referència.

## Ruta de treball

1. Triar variables rellevants per al valor del client.
2. Detectar casos amb dades absents o delicades.
3. Construir una segmentació simple orientada a negoci.
4. Escriure què validaries després amb clustering.

In [1]:
from pathlib import Path
import pandas as pd

CUSTOMERS = Path('..') / 'dades' / 'customers.csv'
customers = pd.read_csv(CUSTOMERS)
customers[['customer_id', 'annual_spend', 'orders_last_12m', 'returns_ratio', 'bike_interest']].head()

,customer_id,annual_spend,orders_last_12m,returns_ratio,bike_interest
0,C00001,4566.25,5,0.0000,electric
1,C00002,3835.08,6,0.0000,electric
2,C00003,1159.47,9,0.0000,urban
3,C00004,2702.07,9,0.1111,electric
4,C00005,304.05,3,0.0000,electric


## 1. Revisió de qualitat

No val segmentar sense mirar abans si hi ha dades que exigixen tractament.

In [2]:
customers.isna().sum().rename('nulls')

customer_id                  0
age                         20
city                         0
channel_preference           0
annual_spend                30
avg_order_value              0
orders_last_12m              0
returns_ratio                0
bike_interest                0
service_interest             0
snapshot_date                0
tenure_months                0
days_since_last_order       20
returned_orders_last_12m     0
Name: nulls, dtype: int64

In [3]:
customers[['annual_spend', 'avg_order_value', 'orders_last_12m', 'returns_ratio']].describe().round(2)

,annual_spend,avg_order_value,orders_last_12m,returns_ratio
count,1970.00,2000.00,2000.00,2000.00
mean,1376.51,254.91,5.25,0.07
std,1618.56,266.43,2.99,0.13
min,35.31,19.06,1.00,0.00
25%,321.96,93.96,3.00,0.00
50%,758.02,145.56,5.00,0.00
75%,1864.55,289.39,7.00,0.10
max,13968.60,1898.82,17.00,1.00


## 2. Segmentació simple orientada a negoci

Mentres no usem clustering formal, podem construir una primera lectura amb regles transparents.

In [4]:
def simple_segment(row):
    if pd.isna(row['annual_spend']):
        return 'revisar_dada'
    if row['annual_spend'] >= 3000 and row['returns_ratio'] <= 0.02:
        return 'premium'
    if row['annual_spend'] >= 1200 and row['orders_last_12m'] >= 8:
        return 'valor_mitja_alt'
    return 'ocasional_o_baix_valor'

customers['simple_segment'] = customers.apply(simple_segment, axis=1)
customers[['customer_id', 'annual_spend', 'orders_last_12m', 'returns_ratio', 'simple_segment']].sort_values(['simple_segment', 'annual_spend'], ascending=[True, False])

,customer_id,annual_spend,orders_last_12m,returns_ratio,simple_segment
1705,C01706,6969.83,7,0.1429,ocasional_o_baix_valor
1071,C01072,6902.40,6,0.1667,ocasional_o_baix_valor
1912,C01913,6602.45,5,0.2000,ocasional_o_baix_valor
1034,C01035,6226.98,6,0.1667,ocasional_o_baix_valor
801,C00802,6005.16,7,0.1429,ocasional_o_baix_valor
...,...,...,...,...,...
1651,C01652,1230.32,8,0.2500,valor_mitja_alt
8,C00009,1219.52,8,0.0000,valor_mitja_alt
915,C00916,1212.24,8,0.1250,valor_mitja_alt
850,C00851,1204.24,8,0.0000,valor_mitja_alt


In [5]:
customers['simple_segment'].value_counts().rename_axis('segment').to_frame('clients')

,clients
segment,
ocasional_o_baix_valor,1510
valor_mitja_alt,285
premium,175
revisar_dada,30


## 3. Lectura per segment

Si la segmentació no permet una interpretació comercial, encara no està madura.

In [6]:
customers.groupby('simple_segment')[['annual_spend', 'avg_order_value', 'orders_last_12m', 'returns_ratio']].mean().round(2)

,annual_spend,avg_order_value,orders_last_12m,returns_ratio
simple_segment,,,,
ocasional_o_baix_valor,806.62,197.21,4.17,0.07
premium,4812.82,770.04,6.75,0.00
revisar_dada,NaN,373.03,4.87,0.06
valor_mitja_alt,2285.92,231.88,10.04,0.07


## 4. Què faltaria validar amb clustering real

- si els grups continuen apareixent sense regles manuals
- si hi ha clients sorollosos o fronterers
- si la lectura comercial es manté quan passem a un algorisme de segmentació

In [7]:
customers[['annual_spend','orders_last_12m','days_since_last_order','returns_ratio']].corr().round(2)

,annual_spend,orders_last_12m,days_since_last_order,returns_ratio
annual_spend,1.00,0.47,-0.18,-0.08
orders_last_12m,0.47,1.00,-0.39,0.02
days_since_last_order,-0.18,-0.39,1.00,-0.04
returns_ratio,-0.08,0.02,-0.04,1.00
